In [1]:
#Joe Neal, Abraham Sharum, Zachary Anderson
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import torch
from torch import nn

df = pd.read_csv("Market.csv",header=None)
df.replace("yes",0,inplace=True)
df.replace("no",0,inplace=True)
df.replace("furnished",0,inplace=True)
df.replace("semi-furnished",0,inplace=True)
df.replace("unfurnished",0,inplace=True)
X = df.drop([0],axis=1)
y = df[0]
avg = int (sum(y)/len(y))

df.fillna(avg,inplace=True)
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

def flatten(x):
    return torch.from_numpy(x.to_numpy()).float().flatten()

def tensify(x):
    return torch.from_numpy(x).float()


x_train, x_test, y_train, y_test = train_test_split(X,y,test_size=.3, shuffle=True)

x_train = x_scaler.fit_transform(x_train)
x_train = tensify(x_train)
x_test = x_scaler.transform(x_test)
x_test = tensify(x_test)

y_train = flatten(y_train)
y_train = y_scaler.fit_transform(y_train.reshape(-1,1))
y_test = flatten(y_test)
y_train = tensify(y_train)
y_test = y_scaler.transform(y_test.reshape(-1,1))
y_test = tensify(y_test)


/tmp/ipykernel_23910/2154112525.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace("no",0,inplace=True)
/tmp/ipykernel_23910/2154112525.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace("unfurnished",0,inplace=True)


In [2]:
class NerualNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(12, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    
    def forward(self,x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [3]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
model = NerualNetwork().to(device)
x_train = x_train.to(device)
y_train = y_train.to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.RAdam(model.parameters(),lr=0.001)
epoch = 50000
for i in range(epoch):
    y_hat = model(x_train).flatten()
    optimizer.zero_grad()
    loss = loss_fn(y_hat,y_train)

    loss.backward()
    optimizer.step()
    if i % 1000 == 0:
        print("Epoch:{:,}".format(i),"\tLoss:{:,}".format(loss.item()))
        

/mnt/AB USB/Machine Learning/Labs/Lab 3/linux/lib/python3.10/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([381, 1])) that is different to the input size (torch.Size([381])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch:0 	Loss:0.17914515733718872
Epoch:1,000 	Loss:0.03083539381623268
Epoch:2,000 	Loss:0.03082636557519436
Epoch:3,000 	Loss:0.03082536719739437
Epoch:4,000 	Loss:0.030824674293398857
Epoch:5,000 	Loss:0.030824260786175728
Epoch:6,000 	Loss:0.03082423098385334
Epoch:7,000 	Loss:0.030824223533272743
Epoch:8,000 	Loss:0.030824219807982445
Epoch:9,000 	Loss:0.030824294313788414
Epoch:10,000 	Loss:0.030824219807982445
Epoch:11,000 	Loss:0.030824219807982445
Epoch:12,000 	Loss:0.03082423098385334
Epoch:13,000 	Loss:0.030824219807982445
Epoch:14,000 	Loss:0.030824273824691772
Epoch:15,000 	Loss:0.030824219807982445
Epoch:16,000 	Loss:0.030824219807982445
Epoch:17,000 	Loss:0.030824219807982445
Epoch:18,000 	Loss:0.030824223533272743
Epoch:19,000 	Loss:0.030824320390820503
Epoch:20,000 	Loss:0.030824294313788414
Epoch:21,000 	Loss:0.03082427754998207
Epoch:22,000 	Loss:0.030824219807982445
Epoch:23,000 	Loss:0.030824219807982445
Epoch:24,000 	Loss:0.030824219807982445
Epoch:25,000 	Loss:0.

In [4]:
if device != "cpu":
    model = model.to("cpu")

with torch.no_grad():
    y_hat = model(x_test).flatten()
    loss = loss_fn(y_hat, y_test)

    print("Test Loss: {:,}".format(loss.item()))
prediction = y_scaler.inverse_transform(y_hat.reshape(-1,1))
true = y_scaler.inverse_transform(y_test.reshape(-1,1))

Test Loss: 0.0336575023829937


/mnt/AB USB/Machine Learning/Labs/Lab 3/linux/lib/python3.10/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([164, 1])) that is different to the input size (torch.Size([164])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [5]:
for i in range(len(true)):
    print(f"True Value:{int (true[i].item())}    Pred Value:{int (prediction[i].item())}")

True Value:3360000    Pred Value:4782038
True Value:3118850    Pred Value:4782083
True Value:3325000    Pred Value:4782061
True Value:3500000    Pred Value:4782059
True Value:6510000    Pred Value:4782112
True Value:2233000    Pred Value:4782096
True Value:8680000    Pred Value:4782174
True Value:4200000    Pred Value:4782167
True Value:9099999    Pred Value:4782184
True Value:5250000    Pred Value:4782118
True Value:4200000    Pred Value:4782165
True Value:6999999    Pred Value:4782118
True Value:4900000    Pred Value:4782084
True Value:6895000    Pred Value:4782176
True Value:4970000    Pred Value:4782213
True Value:7350000    Pred Value:4782238
True Value:3079999    Pred Value:4782074
True Value:3919999    Pred Value:4782077
True Value:6083000    Pred Value:4782158
True Value:4094999    Pred Value:4782116
True Value:5873000    Pred Value:4782148
True Value:6789999    Pred Value:4782115
True Value:5565000    Pred Value:4782046
True Value:4025000    Pred Value:4782161
True Value:48299

In [6]:
count = 0
std_pred = int (np.std(prediction,axis=0).item())
std_true = int (np.std(true,axis=0).item())
sum_true = 0
sum_pred = 0
print("Predicted Standard Deviation:  {:,}".format(std_pred))
print("True Standard Deviation:       {:,}".format(std_true))
r = 1000000
for i in range(len(true)):
    if abs((int (true[i].item()) - (round(int (prediction[i].item())/1000,0) * 1000))) <= r:
        count+=1
    sum_true += int (true[i].item())
    sum_pred += int (round((prediction[i].item())/1000,0) * 1000)
accuracy = round(count/len(true),2)
print("Accuracy within a range of" ,"{:,}:".format(r) ,"{:,}".format(accuracy))
print("True Sum:","{:,}".format(sum_true),"\nPred Sum:", "{:,}".format(sum_pred))

if accuracy > 0.7 and r == 1000000:
    torch.save(model,"model.pth")
    print("NEW HIGH SCORE UPDATE IF CLAUSE")


Predicted Standard Deviation:  1,955
True Standard Deviation:       1,925,674
Accuracy within a range of 1,000,000: 0.35
True Sum: 776,072,787 
Pred Sum: 784,278,000
